### 📈 Why Quants Hate U.S. Equities (and why you should too)

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### Holding a diversified equity portfolio

Just buy VOO or SPY and chill for 30 years

Not a bad plan, on average you'll return maybe 10%, this means

 $$
 P = \frac{1,000,000}{(1 + 0.10)^{30}} \approx \$57,308
 $$

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================================
# LOAD DATA
# ==========================================================

# Your CSV should look like:
# Date,open,high,low,close,volume
# 2000-01-03,148.25,148.25,143.88,145.44,204107.0

df = pd.read_csv("spy_prices.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

dates = df["Date"]
prices = df["close"].astype(float)

if len(df) < 2:
    raise ValueError("Need at least 2 rows to animate the price path.")

# ==========================================================
# SETTINGS
# ==========================================================

N_FRAMES = 180
ANIMATION_SPEED = 18

BG = "#282C34"
PANEL_BG = "#363B44"
GRID = "rgba(255,255,255,0.12)"
TEXT = "#E8E8E8"
MUTED = "#B8B8B8"

PRICE_COLOR = "#00D1FF"
HIGH_LOW_COLOR = "rgba(255,255,255,0.35)"

PRICE_WIDTH = 2.8
MARKER_SIZE = 7

# ==========================================================
# STATS
# ==========================================================

def max_drawdown(values):
    values = pd.Series(values).astype(float)
    running_max = values.cummax()
    drawdown = values / running_max - 1
    return drawdown.min()


def stats_for(price_slice):
    price_slice = pd.Series(price_slice).astype(float)
    rets = price_slice.pct_change().dropna()

    start_price = price_slice.iloc[0]
    end_price = price_slice.iloc[-1]

    total_return = end_price / start_price - 1 if start_price != 0 else 0.0
    high_price = price_slice.max()
    low_price = price_slice.min()
    mdd = max_drawdown(price_slice)

    vol = rets.std(ddof=1) * np.sqrt(252) if len(rets) > 1 else 0.0

    return {
        "start": start_price,
        "last": end_price,
        "high": high_price,
        "low": low_price,
        "total_return": total_return,
        "ann_vol": vol,
        "mdd": mdd,
    }


def make_stats_table(stats):
    return go.Table(
        columnwidth=[1.4, 1.0],
        header=dict(
            values=["Metric", "Value"],
            fill_color=PANEL_BG,
            line_color="rgba(255,255,255,0.18)",
            font=dict(color=TEXT, size=14),
            align=["left", "right"],
            height=38,
        ),
        cells=dict(
            values=[
                [
                    "Start Price",
                    "Last Price",
                    "High",
                    "Low",
                    "Total Return",
                    "Annualized Volatility",
                    "Max Drawdown",
                ],
                [
                    f"${stats['start']:,.2f}",
                    f"${stats['last']:,.2f}",
                    f"${stats['high']:,.2f}",
                    f"${stats['low']:,.2f}",
                    f"{stats['total_return']:.2%}",
                    f"{stats['ann_vol']:.2%}",
                    f"{stats['mdd']:.2%}",
                ],
            ],
            fill_color=[
                ["#363B44"] * 7,
                ["#363B44"] * 7,
            ],
            line_color="rgba(255,255,255,0.10)",
            font=dict(
                color=[
                    [MUTED] * 7,
                    [PRICE_COLOR] * 7,
                ],
                size=13,
            ),
            align=["left", "right"],
            height=42,
        ),
    )


# ==========================================================
# FRAME INDEXES
# ==========================================================

if len(dates) <= N_FRAMES:
    frame_indices = list(range(len(dates)))
else:
    frame_indices = np.linspace(0, len(dates) - 1, N_FRAMES).astype(int)
    frame_indices = sorted(set(frame_indices.tolist()))

if frame_indices[-1] != len(dates) - 1:
    frame_indices.append(len(dates) - 1)

# ==========================================================
# AXIS SCALING
# ==========================================================

x_min = dates.iloc[0]
x_max = dates.iloc[-1]

y_padding = (prices.max() - prices.min()) * 0.12
if y_padding == 0:
    y_padding = prices.iloc[0] * 0.05

y_min = prices.min() - y_padding
y_max = prices.max() + y_padding

# ==========================================================
# ANNOTATIONS
# ==========================================================

def price_annotation(i=0):
    start_price = prices.iloc[0]
    current_price = prices.iloc[i]
    current_return = current_price / start_price - 1 if start_price != 0 else 0.0

    return dict(
        x=0.075,
        y=0.865,
        xref="paper",
        yref="paper",
        showarrow=False,
        align="left",
        bgcolor="rgba(40,44,52,0.84)",
        bordercolor="rgba(255,255,255,0.18)",
        borderwidth=1,
        borderpad=8,
        font=dict(color=TEXT, size=14),
        text=(
            "<b>Price Path</b><br>"
            f"Current close: <span style='color:{PRICE_COLOR}'>${current_price:,.2f}</span><br>"
            f"Return since start: {current_return:.2%}"
        ),
    )


def date_annotation(i=0):
    return dict(
        x=0.985,
        y=-0.14,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(color=TEXT, size=15),
        text=f"Date: {dates.iloc[i].strftime('%b %d, %Y')}",
    )


# ==========================================================
# FIGURE
# ==========================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.72, 0.28],
    horizontal_spacing=0.07,
    specs=[[{"type": "xy"}, {"type": "table"}]],
)

# Trace 0: price line
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[prices.iloc[0]],
        mode="lines",
        line=dict(color=PRICE_COLOR, width=PRICE_WIDTH),
        name="Close price",
        hovertemplate="Date: %{x|%b %d, %Y}<br>Close: $%{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Trace 1: endpoint marker
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0]],
        y=[prices.iloc[0]],
        mode="markers",
        marker=dict(size=MARKER_SIZE, color=PRICE_COLOR),
        showlegend=False,
        hovertemplate="Date: %{x|%b %d, %Y}<br>Close: $%{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# Trace 2: high-low vertical bar for current day
fig.add_trace(
    go.Scatter(
        x=[dates.iloc[0], dates.iloc[0]],
        y=[df["low"].iloc[0], df["high"].iloc[0]],
        mode="lines",
        line=dict(color=HIGH_LOW_COLOR, width=1.6),
        name="Current day high-low",
        hovertemplate="Current day range<extra></extra>",
    ),
    row=1,
    col=1,
)

# Trace 3: stats table
fig.add_trace(
    make_stats_table(stats_for(prices.iloc[:1])),
    row=1,
    col=2,
)

# ==========================================================
# FRAMES
# ==========================================================

frames = []
slider_steps = []

for i in frame_indices:
    frame_name = f"frame_{i}"

    price_slice = prices.iloc[: i + 1]
    current_stats = stats_for(price_slice)

    frames.append(
        go.Frame(
            name=frame_name,
            traces=[0, 1, 2, 3],
            data=[
                go.Scatter(
                    x=dates.iloc[: i + 1],
                    y=prices.iloc[: i + 1],
                ),
                go.Scatter(
                    x=[dates.iloc[i]],
                    y=[prices.iloc[i]],
                ),
                go.Scatter(
                    x=[dates.iloc[i], dates.iloc[i]],
                    y=[df["low"].iloc[i], df["high"].iloc[i]],
                ),
                make_stats_table(current_stats),
            ],
            layout=go.Layout(
                annotations=[
                    price_annotation(i),
                    date_annotation(i),
                ]
            ),
        )
    )

    slider_steps.append(
        dict(
            args=[
                [frame_name],
                dict(
                    frame=dict(duration=0, redraw=True),
                    transition=dict(duration=0),
                    mode="immediate",
                ),
            ],
            label=dates.iloc[i].strftime("%Y"),
            method="animate",
        )
    )

fig.frames = frames

# ==========================================================
# LAYOUT
# ==========================================================

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    width=1250,
    height=720,
    margin=dict(l=70, r=45, t=105, b=120),
    showlegend=True,
    legend=dict(
        orientation="h",
        x=0.03,
        y=1.04,
        xanchor="left",
        yanchor="bottom",
        bgcolor="rgba(0,0,0,0)",
        font=dict(color=TEXT, size=13),
        itemwidth=30,
    ),
    title=dict(
        text="SPY Price Path",
        x=0.04,
        y=0.97,
        xanchor="left",
        font=dict(size=27, color=TEXT),
    ),
    annotations=[
        price_annotation(0),
        date_annotation(0),
    ],
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            showactive=False,
            x=0.00,
            y=-0.16,
            xanchor="left",
            yanchor="top",
            pad=dict(r=10, t=30),
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(duration=ANIMATION_SPEED, redraw=True),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                ),
                dict(
                    label="⏸ Pause",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            frame=dict(duration=0, redraw=True),
                            transition=dict(duration=0),
                            mode="immediate",
                        ),
                    ],
                ),
            ],
        )
    ],
    sliders=[
        dict(
            active=0,
            x=0.19,
            y=-0.14,
            len=0.75,
            xanchor="left",
            yanchor="top",
            pad=dict(t=45, b=10),
            currentvalue=dict(visible=False),
            transition=dict(duration=0),
            steps=slider_steps,
        )
    ],
)

# ==========================================================
# AXES
# ==========================================================

fig.update_xaxes(
    row=1,
    col=1,
    range=[x_min, x_max],
    title_text="Date",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_yaxes(
    row=1,
    col=1,
    range=[y_min, y_max],
    title_text="Price ($)",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.show()

# Optional: save to standalone HTML
# fig.write_html("animated_price_path.html", auto_open=True)

###### ______________________________________________________________________________________________________________________________________

##### When you buy makes an incredible impact

There's roughly 6 million ways to enter a trade on a one hour candle in a reasonably liquid instrument

No kidding when you buy is going to make a tremendous impact, it isn't even about volatility drag yet it's just about your cost basis

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================================
# LOAD DATA
# ==========================================================

df = pd.read_csv("spy_prices.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df.columns = [c.lower() for c in df.columns]
df = df.rename(columns={"date": "Date"})

if "Date" not in df.columns:
    raise ValueError("CSV must contain a Date column.")

if "close" not in df.columns:
    raise ValueError("CSV must contain a close column.")

df["close"] = df["close"].astype(float)

dates = df["Date"]
prices = df["close"]

# ==========================================================
# SETTINGS
# ==========================================================

START_CAPITAL = 100_000

ENTRY_DATE_OLD = pd.Timestamp("2022-01-01")
ENTRY_DATE_NEW = pd.Timestamp("2023-01-01")

ENTRY_LABEL_OLD = "Jan 2022"
ENTRY_LABEL_NEW = "Jan 2023"

N_FRAMES = 180
ANIMATION_SPEED = 20

BG = "#282C34"
PANEL_BG = "#363B44"
GRID = "rgba(255,255,255,0.12)"
TEXT = "#E8E8E8"
MUTED = "#B8B8B8"

PRICE_COLOR = "#00D1FF"
OLD_COLOR = "#00D1FF"
NEW_COLOR = "#FFB000"
GAP_COLOR = "rgba(255,255,255,0.45)"

PRICE_WIDTH = 2.8
BASIS_WIDTH = 1.9
MARKER_SIZE = 8

# ==========================================================
# ENTRY DATES
# ==========================================================

def first_trading_day_on_or_after(target_date):
    eligible = df[df["Date"] >= target_date]

    if eligible.empty:
        raise ValueError(f"No available trading date on or after {target_date.date()}.")

    return eligible.index[0]


old_entry_idx = first_trading_day_on_or_after(ENTRY_DATE_OLD)
new_entry_idx = first_trading_day_on_or_after(ENTRY_DATE_NEW)

old_entry_date = dates.iloc[old_entry_idx]
new_entry_date = dates.iloc[new_entry_idx]

old_cost_basis = prices.iloc[old_entry_idx]
new_cost_basis = prices.iloc[new_entry_idx]

old_shares = START_CAPITAL / old_cost_basis
new_shares = START_CAPITAL / new_cost_basis

# ==========================================================
# PORTFOLIO SERIES
# ==========================================================

old_value = pd.Series(np.nan, index=df.index, dtype=float)
new_value = pd.Series(np.nan, index=df.index, dtype=float)

old_value.iloc[old_entry_idx:] = old_shares * prices.iloc[old_entry_idx:]
new_value.iloc[new_entry_idx:] = new_shares * prices.iloc[new_entry_idx:]

# ==========================================================
# STATS
# ==========================================================

def max_drawdown(values):
    values = pd.Series(values).dropna().astype(float)

    if len(values) == 0:
        return np.nan

    running_max = values.cummax()
    drawdown = values / running_max - 1
    return drawdown.min()


def stats_for(i):
    current_price = prices.iloc[i]

    old_active = i >= old_entry_idx
    new_active = i >= new_entry_idx

    old_terminal = old_value.iloc[i] if old_active else np.nan
    new_terminal = new_value.iloc[i] if new_active else np.nan

    old_ret = old_terminal / START_CAPITAL - 1 if old_active else np.nan
    new_ret = new_terminal / START_CAPITAL - 1 if new_active else np.nan

    value_gap = (
        old_terminal - new_terminal
        if old_active and new_active
        else np.nan
    )

    basis_gap = new_cost_basis - old_cost_basis
    basis_gap_pct = basis_gap / old_cost_basis

    old_mdd = max_drawdown(old_value.iloc[old_entry_idx : i + 1]) if old_active else np.nan
    new_mdd = max_drawdown(new_value.iloc[new_entry_idx : i + 1]) if new_active else np.nan

    return {
        "current_price": current_price,
        "old_terminal": old_terminal,
        "new_terminal": new_terminal,
        "old_return": old_ret,
        "new_return": new_ret,
        "value_gap": value_gap,
        "basis_gap": basis_gap,
        "basis_gap_pct": basis_gap_pct,
        "old_mdd": old_mdd,
        "new_mdd": new_mdd,
    }


def money_or_dash(x):
    if pd.isna(x):
        return "—"
    return f"${x:,.0f}"


def money2_or_dash(x):
    if pd.isna(x):
        return "—"
    return f"${x:,.2f}"


def pct_or_dash(x):
    if pd.isna(x):
        return "—"
    return f"{x:.2%}"


def make_stats_table(s):
    return go.Table(
        columnwidth=[1.45, 1.0, 1.0],
        header=dict(
            values=["Metric", ENTRY_LABEL_OLD, ENTRY_LABEL_NEW],
            fill_color=PANEL_BG,
            line_color="rgba(255,255,255,0.18)",
            font=dict(color=TEXT, size=14),
            align=["left", "right", "right"],
            height=38,
        ),
        cells=dict(
            values=[
                [
                    "Entry Date",
                    "Cost Basis",
                    "Shares Bought",
                    "Current Value",
                    "Total Return",
                    "Max Drawdown",
                ],
                [
                    old_entry_date.strftime("%b %d, %Y"),
                    money2_or_dash(old_cost_basis),
                    f"{old_shares:,.2f}",
                    money_or_dash(s["old_terminal"]),
                    pct_or_dash(s["old_return"]),
                    pct_or_dash(s["old_mdd"]),
                ],
                [
                    new_entry_date.strftime("%b %d, %Y"),
                    money2_or_dash(new_cost_basis),
                    f"{new_shares:,.2f}",
                    money_or_dash(s["new_terminal"]),
                    pct_or_dash(s["new_return"]),
                    pct_or_dash(s["new_mdd"]),
                ],
            ],
            fill_color=[
                ["#363B44"] * 6,
                ["#363B44"] * 6,
                ["#363B44"] * 6,
            ],
            line_color="rgba(255,255,255,0.10)",
            font=dict(
                color=[
                    [MUTED] * 6,
                    [OLD_COLOR] * 6,
                    [NEW_COLOR] * 6,
                ],
                size=13,
            ),
            align=["left", "right", "right"],
            height=42,
        ),
    )


# ==========================================================
# FRAME INDEXES
# ==========================================================

start_idx = old_entry_idx
end_idx = len(df) - 1

available_indices = list(range(start_idx, end_idx + 1))

if len(available_indices) <= N_FRAMES:
    frame_indices = available_indices
else:
    frame_indices = np.linspace(start_idx, end_idx, N_FRAMES).astype(int)
    frame_indices = sorted(set(frame_indices.tolist()))

if frame_indices[-1] != end_idx:
    frame_indices.append(end_idx)

# ==========================================================
# AXIS SCALING
# ==========================================================

x_min = dates.iloc[start_idx]
x_max = dates.iloc[end_idx]

visible_prices = prices.iloc[start_idx : end_idx + 1]
y_padding = (visible_prices.max() - visible_prices.min()) * 0.12

if y_padding == 0:
    y_padding = visible_prices.iloc[0] * 0.05

y_min = visible_prices.min() - y_padding
y_max = visible_prices.max() + y_padding

# ==========================================================
# ANNOTATIONS
# ==========================================================

def main_annotation(i):
    s = stats_for(i)

    if i < new_entry_idx:
        gap_text = f"{ENTRY_LABEL_NEW} portfolio has not entered yet."
    else:
        gap_text = (
            f"Terminal value gap: "
            f"<span style='color:{OLD_COLOR}'>${s['value_gap']:,.0f}</span>"
        )

    return dict(
        x=0.055,
        y=0.89,
        xref="paper",
        yref="paper",
        showarrow=False,
        align="left",
        bgcolor="rgba(40,44,52,0.86)",
        bordercolor="rgba(255,255,255,0.18)",
        borderwidth=1,
        borderpad=8,
        font=dict(color=TEXT, size=14),
        text=(
            "<b>Vol Drag / Entry Timing Impact</b><br>"
            f"SPY close: <span style='color:{PRICE_COLOR}'>${s['current_price']:,.2f}</span><br>"
            f"Cost basis gap: ${s['basis_gap']:,.2f} "
            f"({s['basis_gap_pct']:.2%})<br>"
            f"{gap_text}"
        ),
    )


def date_annotation(i):
    return dict(
        x=0.985,
        y=-0.14,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(color=TEXT, size=15),
        text=f"Date: {dates.iloc[i].strftime('%b %d, %Y')}",
    )


# ==========================================================
# FIGURE
# ==========================================================

fig = make_subplots(
    rows=2,
    cols=2,
    column_widths=[0.70, 0.30],
    row_heights=[0.64, 0.36],
    horizontal_spacing=0.07,
    vertical_spacing=0.13,
    specs=[
        [{"type": "xy", "rowspan": 2}, {"type": "table"}],
        [None, {"type": "xy"}],
    ],
    subplot_titles=[
        "",
        "",
        "",
        "Cost Basis vs Terminal Value",
    ],
)

# ==========================================================
# LEFT CHART: SPY PRICE PATH
# ==========================================================

fig.add_trace(
    go.Scatter(
        x=[dates.iloc[start_idx]],
        y=[prices.iloc[start_idx]],
        mode="lines",
        line=dict(color=PRICE_COLOR, width=PRICE_WIDTH),
        name="SPY close",
        hovertemplate="Date: %{x|%b %d, %Y}<br>SPY: $%{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=[dates.iloc[start_idx]],
        y=[prices.iloc[start_idx]],
        mode="markers",
        marker=dict(size=MARKER_SIZE, color=PRICE_COLOR),
        showlegend=False,
        hovertemplate="Date: %{x|%b %d, %Y}<br>SPY: $%{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=[old_entry_date, x_max],
        y=[old_cost_basis, old_cost_basis],
        mode="lines",
        line=dict(color=OLD_COLOR, width=BASIS_WIDTH, dash="dash"),
        name=f"{ENTRY_LABEL_OLD} cost basis",
        hovertemplate=f"{ENTRY_LABEL_OLD} cost basis: ${old_cost_basis:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=[new_entry_date, new_entry_date],
        y=[new_cost_basis, new_cost_basis],
        mode="lines",
        line=dict(color=NEW_COLOR, width=BASIS_WIDTH, dash="dash"),
        name=f"{ENTRY_LABEL_NEW} cost basis",
        hovertemplate=f"{ENTRY_LABEL_NEW} cost basis: ${new_cost_basis:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=[new_entry_date, new_entry_date],
        y=[old_cost_basis, new_cost_basis],
        mode="lines",
        line=dict(color=GAP_COLOR, width=1.8, dash="dot"),
        name="Cost basis gap",
        hovertemplate="Cost basis gap<extra></extra>",
    ),
    row=1,
    col=1,
)

# ==========================================================
# RIGHT TOP: TABLE
# ==========================================================

fig.add_trace(
    make_stats_table(stats_for(start_idx)),
    row=1,
    col=2,
)

# ==========================================================
# RIGHT BOTTOM: BAR CHART
# ==========================================================

initial_stats = stats_for(start_idx)

fig.add_trace(
    go.Bar(
        x=[ENTRY_LABEL_OLD, ENTRY_LABEL_NEW],
        y=[old_cost_basis, new_cost_basis],
        name="Cost basis",
        marker=dict(color=[OLD_COLOR, NEW_COLOR]),
        opacity=0.55,
        hovertemplate="%{x}<br>Cost basis: $%{y:.2f}<extra></extra>",
    ),
    row=2,
    col=2,
)

fig.add_trace(
    go.Bar(
        x=[ENTRY_LABEL_OLD, ENTRY_LABEL_NEW],
        y=[initial_stats["old_terminal"], 0],
        name="Terminal value",
        marker=dict(color=[OLD_COLOR, NEW_COLOR]),
        opacity=1.0,
        hovertemplate="%{x}<br>Terminal value: $%{y:,.0f}<extra></extra>",
    ),
    row=2,
    col=2,
)

# Trace indexes:
# 0 SPY close
# 1 endpoint marker
# 2 old basis line
# 3 new basis line
# 4 basis gap
# 5 table
# 6 cost basis bars
# 7 terminal value bars

# ==========================================================
# FRAMES
# ==========================================================

frames = []
slider_steps = []

for i in frame_indices:
    frame_name = f"frame_{i}"
    s = stats_for(i)

    price_x = dates.iloc[start_idx : i + 1]
    price_y = prices.iloc[start_idx : i + 1]

    if i >= new_entry_idx:
        new_basis_x = [new_entry_date, dates.iloc[i]]
        new_basis_y = [new_cost_basis, new_cost_basis]
        new_terminal_bar = s["new_terminal"]
    else:
        new_basis_x = [new_entry_date, new_entry_date]
        new_basis_y = [new_cost_basis, new_cost_basis]
        new_terminal_bar = 0

    frames.append(
        go.Frame(
            name=frame_name,
            traces=[0, 1, 2, 3, 4, 5, 6, 7],
            data=[
                go.Scatter(
                    x=price_x,
                    y=price_y,
                ),
                go.Scatter(
                    x=[dates.iloc[i]],
                    y=[prices.iloc[i]],
                ),
                go.Scatter(
                    x=[old_entry_date, dates.iloc[i]],
                    y=[old_cost_basis, old_cost_basis],
                ),
                go.Scatter(
                    x=new_basis_x,
                    y=new_basis_y,
                ),
                go.Scatter(
                    x=[new_entry_date, new_entry_date],
                    y=[old_cost_basis, new_cost_basis],
                ),
                make_stats_table(s),
                go.Bar(
                    x=[ENTRY_LABEL_OLD, ENTRY_LABEL_NEW],
                    y=[old_cost_basis, new_cost_basis],
                ),
                go.Bar(
                    x=[ENTRY_LABEL_OLD, ENTRY_LABEL_NEW],
                    y=[
                        s["old_terminal"],
                        new_terminal_bar,
                    ],
                ),
            ],
            layout=go.Layout(
                annotations=[
                    main_annotation(i),
                    date_annotation(i),
                ]
            ),
        )
    )

    slider_steps.append(
        dict(
            args=[
                [frame_name],
                dict(
                    frame=dict(duration=0, redraw=True),
                    transition=dict(duration=0),
                    mode="immediate",
                ),
            ],
            label=dates.iloc[i].strftime("%Y"),
            method="animate",
        )
    )

fig.frames = frames

# ==========================================================
# LAYOUT
# ==========================================================

right_bar_max = max(
    old_cost_basis,
    new_cost_basis,
    np.nanmax(old_value),
    np.nanmax(new_value),
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    width=1350,
    height=780,
    margin=dict(l=70, r=45, t=105, b=120),
    barmode="group",
    showlegend=True,
    legend=dict(
        orientation="h",
        x=0.03,
        y=1.04,
        xanchor="left",
        yanchor="bottom",
        bgcolor="rgba(0,0,0,0)",
        font=dict(color=TEXT, size=13),
        itemwidth=30,
    ),
    title=dict(
        text="SPY Entry Timing: Jan 2022 vs Jan 2023",
        x=0.04,
        y=0.97,
        xanchor="left",
        font=dict(size=27, color=TEXT),
    ),
    annotations=[
        main_annotation(start_idx),
        date_annotation(start_idx),
    ],
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            showactive=False,
            x=0.00,
            y=-0.16,
            xanchor="left",
            yanchor="top",
            pad=dict(r=10, t=30),
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(duration=ANIMATION_SPEED, redraw=True),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                ),
                dict(
                    label="⏸ Pause",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            frame=dict(duration=0, redraw=True),
                            transition=dict(duration=0),
                            mode="immediate",
                        ),
                    ],
                ),
            ],
        )
    ],
    sliders=[
        dict(
            active=0,
            x=0.19,
            y=-0.14,
            len=0.75,
            xanchor="left",
            yanchor="top",
            pad=dict(t=45, b=10),
            currentvalue=dict(visible=False),
            transition=dict(duration=0),
            steps=slider_steps,
        )
    ],
)

# ==========================================================
# AXES
# ==========================================================

fig.update_xaxes(
    row=1,
    col=1,
    range=[x_min, x_max],
    title_text="Date",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_yaxes(
    row=1,
    col=1,
    range=[y_min, y_max],
    title_text="SPY Price ($)",
    title_font=dict(size=16, color=TEXT),
    tickfont=dict(size=12, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_xaxes(
    row=2,
    col=2,
    tickfont=dict(size=12, color=TEXT),
    showgrid=False,
    linecolor=TEXT,
)

fig.update_yaxes(
    row=2,
    col=2,
    range=[0, right_bar_max * 1.15],
    title_text="$",
    title_font=dict(size=14, color=TEXT),
    tickfont=dict(size=11, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_xaxes(row=1, col=2, visible=False)
fig.update_yaxes(row=1, col=2, visible=False)

fig.show()

# Optional
# fig.write_html("spy_entry_timing_2022_vs_2023.html", auto_open=True)

###### ______________________________________________________________________________________________________________________________________

##### You're not timing sh*t

It isn't about timing *anything* it's about positioning and survival

Creating effective positioning enables the safe use of leverage to generate outsized risk-adjusted returns and lower volatility drag

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================================
# LOAD DATA
# ==========================================================

df = pd.read_csv("spy_prices.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df.columns = [c.lower() for c in df.columns]
df = df.rename(columns={"date": "Date"})

if "Date" not in df.columns:
    raise ValueError("CSV must contain a Date column.")

if "close" not in df.columns:
    raise ValueError("CSV must contain a close column.")

df["close"] = df["close"].astype(float)

dates = df["Date"]
prices = df["close"]

# ==========================================================
# SETTINGS
# ==========================================================

START_CAPITAL = 100_000

ENTRY_DATE_JAN_2022 = pd.Timestamp("2022-01-01")
ENTRY_DATE_JAN_2023 = pd.Timestamp("2023-01-01")

LABEL_2022 = "Jan 2022 B&H"
LABEL_2023 = "Jan 2023 Timing"
LABEL_CONVEXITY = "Jan 2022 + Put Reinvest"

# ==========================================================
# CONVEXITY ASSUMPTIONS
# ==========================================================

PUT_PREMIUM_PCT_OF_CAPITAL = 0.025       # 2.5% spent on put premium
PUT_STRIKE_PCT_OF_ENTRY = 1.00           # ATM put
PUT_NOTIONAL_PCT_OF_STOCK = 1.00         # 100% hedge on stock sleeve
REINVEST_PUT_PROCEEDS = True             # Net put profits buy more SPY

# Animation
N_FRAMES = 180
ANIMATION_SPEED = 20

# Styling
BG = "#282C34"
PANEL_BG = "#363B44"
GRID = "rgba(255,255,255,0.12)"
TEXT = "#E8E8E8"
MUTED = "#B8B8B8"

PRICE_COLOR = "#B8B8B8"
BH_COLOR = "#00D1FF"
TIMING_COLOR = "#FFB000"
CONVEXITY_COLOR = "#00FF9C"

PRICE_WIDTH = 2.6
BASIS_WIDTH = 1.9
PORTFOLIO_WIDTH = 2.9
MARKER_SIZE = 8

# ==========================================================
# ENTRY DATE HELPERS
# ==========================================================

def first_trading_day_on_or_after(target_date):
    eligible = df[df["Date"] >= target_date]

    if eligible.empty:
        raise ValueError(f"No available trading date on or after {target_date.date()}.")

    return eligible.index[0]


idx_2022 = first_trading_day_on_or_after(ENTRY_DATE_JAN_2022)
idx_2023 = first_trading_day_on_or_after(ENTRY_DATE_JAN_2023)

entry_date_2022 = dates.iloc[idx_2022]
entry_date_2023 = dates.iloc[idx_2023]

basis_2022 = prices.iloc[idx_2022]
basis_2023 = prices.iloc[idx_2023]

# ==========================================================
# BASE PORTFOLIOS
# ==========================================================

shares_2022 = START_CAPITAL / basis_2022
shares_2023 = START_CAPITAL / basis_2023

value_2022 = pd.Series(np.nan, index=df.index, dtype=float)
value_2023 = pd.Series(np.nan, index=df.index, dtype=float)

value_2022.iloc[idx_2022:] = shares_2022 * prices.iloc[idx_2022:]
value_2023.iloc[idx_2023:] = shares_2023 * prices.iloc[idx_2023:]

# ==========================================================
# CONVEXITY PORTFOLIO — FIXED ACCOUNTING
# ==========================================================

put_premium_dollars = START_CAPITAL * PUT_PREMIUM_PCT_OF_CAPITAL
initial_stock_capital = START_CAPITAL - put_premium_dollars

convexity_initial_shares = initial_stock_capital / basis_2022

put_strike = basis_2022 * PUT_STRIKE_PCT_OF_ENTRY
put_units = convexity_initial_shares * PUT_NOTIONAL_PCT_OF_STOCK


def put_value_for_price(price):
    """
    Simple intrinsic-value long put proxy.

    This is not Black-Scholes.
    It intentionally isolates downside convexity using only SPY prices.
    """
    return max(put_strike - price, 0) * put_units


raw_put_value = prices.apply(put_value_for_price)
raw_put_value.iloc[:idx_2022] = np.nan

# Monetized put value increases only when the put reaches a new high.
# This represents harvesting convexity during the drawdown.
monetized_put_value = raw_put_value.copy()
monetized_put_value.iloc[idx_2022:] = raw_put_value.iloc[idx_2022:].cummax()

# Gross increase in monetized put value.
incremental_put_cash = monetized_put_value.diff().fillna(0)
incremental_put_cash = incremental_put_cash.clip(lower=0)

convexity_shares = pd.Series(np.nan, index=df.index, dtype=float)
convexity_value = pd.Series(np.nan, index=df.index, dtype=float)
effective_convexity_basis = pd.Series(np.nan, index=df.index, dtype=float)
gross_put_cash_realized = pd.Series(np.nan, index=df.index, dtype=float)
net_put_profit_realized = pd.Series(np.nan, index=df.index, dtype=float)
reinvested_put_cash = pd.Series(np.nan, index=df.index, dtype=float)

current_shares = convexity_initial_shares

cumulative_gross_put_cash = 0.0
cumulative_reinvested_put_cash = 0.0

for i in range(idx_2022, len(df)):
    price = prices.iloc[i]

    new_put_cash = incremental_put_cash.iloc[i]

    prior_gross_put_cash = cumulative_gross_put_cash
    candidate_gross_put_cash = cumulative_gross_put_cash + new_put_cash

    # Conservative accounting:
    # The put must first earn back the original premium.
    # Only put gains above the premium are reinvested into SPY.
    prior_reinvestable_cash = max(prior_gross_put_cash - put_premium_dollars, 0)
    candidate_reinvestable_cash = max(candidate_gross_put_cash - put_premium_dollars, 0)

    new_reinvestable_cash = candidate_reinvestable_cash - prior_reinvestable_cash

    if REINVEST_PUT_PROCEEDS and new_reinvestable_cash > 0:
        additional_shares = new_reinvestable_cash / price
        current_shares += additional_shares
        cumulative_reinvested_put_cash += new_reinvestable_cash

    cumulative_gross_put_cash = candidate_gross_put_cash

    convexity_shares.iloc[i] = current_shares
    gross_put_cash_realized.iloc[i] = cumulative_gross_put_cash
    net_put_profit_realized.iloc[i] = cumulative_gross_put_cash - put_premium_dollars
    reinvested_put_cash.iloc[i] = cumulative_reinvested_put_cash

    # Portfolio value is only shares * price.
    # Reinvested put cash is already embedded via the increased share count.
    convexity_value.iloc[i] = current_shares * price

    # Correct effective basis:
    # total original capital divided by current shares.
    # Do not subtract put proceeds again, or basis gets double-counted too low.
    effective_convexity_basis.iloc[i] = START_CAPITAL / current_shares

# ==========================================================
# STATS HELPERS
# ==========================================================

def max_drawdown(values):
    values = pd.Series(values).dropna().astype(float)

    if len(values) == 0:
        return np.nan

    running_max = values.cummax()
    drawdown = values / running_max - 1
    return drawdown.min()


def money_or_dash(x):
    if pd.isna(x):
        return "—"
    return f"${x:,.0f}"


def money2_or_dash(x):
    if pd.isna(x):
        return "—"
    return f"${x:,.2f}"


def pct_or_dash(x):
    if pd.isna(x):
        return "—"
    return f"{x:.2%}"


def number_or_dash(x):
    if pd.isna(x):
        return "—"
    return f"{x:,.2f}"


def stats_for(i):
    timing_active = i >= idx_2023

    bh_terminal = value_2022.iloc[i]
    timing_terminal = value_2023.iloc[i] if timing_active else np.nan
    convexity_terminal = convexity_value.iloc[i]

    bh_return = bh_terminal / START_CAPITAL - 1
    timing_return = timing_terminal / START_CAPITAL - 1 if timing_active else np.nan
    convexity_return = convexity_terminal / START_CAPITAL - 1

    convexity_vs_timing_gap = (
        convexity_terminal - timing_terminal
        if timing_active
        else np.nan
    )

    convexity_vs_bh_gap = convexity_terminal - bh_terminal

    return {
        "price": prices.iloc[i],

        "bh_terminal": bh_terminal,
        "timing_terminal": timing_terminal,
        "convexity_terminal": convexity_terminal,

        "bh_return": bh_return,
        "timing_return": timing_return,
        "convexity_return": convexity_return,

        "bh_mdd": max_drawdown(value_2022.iloc[idx_2022 : i + 1]),
        "timing_mdd": max_drawdown(value_2023.iloc[idx_2023 : i + 1]) if timing_active else np.nan,
        "convexity_mdd": max_drawdown(convexity_value.iloc[idx_2022 : i + 1]),

        "convexity_shares": convexity_shares.iloc[i],
        "effective_basis": effective_convexity_basis.iloc[i],

        "gross_put_cash": gross_put_cash_realized.iloc[i],
        "net_put_profit": net_put_profit_realized.iloc[i],
        "reinvested_put_cash": reinvested_put_cash.iloc[i],

        "convexity_vs_timing_gap": convexity_vs_timing_gap,
        "convexity_vs_bh_gap": convexity_vs_bh_gap,
    }


def make_stats_table(s):
    return go.Table(
        columnwidth=[1.35, 1.0, 1.0, 1.0],
        header=dict(
            values=["Metric", LABEL_2022, LABEL_2023, LABEL_CONVEXITY],
            fill_color=PANEL_BG,
            line_color="rgba(255,255,255,0.18)",
            font=dict(color=TEXT, size=13),
            align=["left", "right", "right", "right"],
            height=38,
        ),
        cells=dict(
            values=[
                [
                    "Entry Date",
                    "Initial Basis",
                    "Effective Basis",
                    "Shares Owned",
                    "Put Cash Reinvested",
                    "Current Value",
                    "Total Return",
                    "Max Drawdown",
                ],
                [
                    entry_date_2022.strftime("%b %d, %Y"),
                    money2_or_dash(basis_2022),
                    money2_or_dash(basis_2022),
                    number_or_dash(shares_2022),
                    "—",
                    money_or_dash(s["bh_terminal"]),
                    pct_or_dash(s["bh_return"]),
                    pct_or_dash(s["bh_mdd"]),
                ],
                [
                    entry_date_2023.strftime("%b %d, %Y"),
                    money2_or_dash(basis_2023),
                    money2_or_dash(basis_2023),
                    number_or_dash(shares_2023),
                    "—",
                    money_or_dash(s["timing_terminal"]),
                    pct_or_dash(s["timing_return"]),
                    pct_or_dash(s["timing_mdd"]),
                ],
                [
                    entry_date_2022.strftime("%b %d, %Y"),
                    money2_or_dash(basis_2022),
                    money2_or_dash(s["effective_basis"]),
                    number_or_dash(s["convexity_shares"]),
                    money_or_dash(s["reinvested_put_cash"]),
                    money_or_dash(s["convexity_terminal"]),
                    pct_or_dash(s["convexity_return"]),
                    pct_or_dash(s["convexity_mdd"]),
                ],
            ],
            fill_color=[
                ["#363B44"] * 8,
                ["#363B44"] * 8,
                ["#363B44"] * 8,
                ["#363B44"] * 8,
            ],
            line_color="rgba(255,255,255,0.10)",
            font=dict(
                color=[
                    [MUTED] * 8,
                    [BH_COLOR] * 8,
                    [TIMING_COLOR] * 8,
                    [CONVEXITY_COLOR] * 8,
                ],
                size=12,
            ),
            align=["left", "right", "right", "right"],
            height=38,
        ),
    )

# ==========================================================
# FRAME INDEXES
# ==========================================================

start_idx = idx_2022
end_idx = len(df) - 1

if len(df.iloc[start_idx : end_idx + 1]) <= N_FRAMES:
    frame_indices = list(range(start_idx, end_idx + 1))
else:
    frame_indices = np.linspace(start_idx, end_idx, N_FRAMES).astype(int)
    frame_indices = sorted(set(frame_indices.tolist()))

if frame_indices[-1] != end_idx:
    frame_indices.append(end_idx)

# ==========================================================
# YEARLY SLIDER STEPS ONLY
# ==========================================================

year_step_indices = []
seen_years = set()

for i in frame_indices:
    year = dates.iloc[i].year

    if year not in seen_years:
        year_step_indices.append(i)
        seen_years.add(year)

if end_idx not in year_step_indices:
    year_step_indices.append(end_idx)

# ==========================================================
# AXIS SCALING
# ==========================================================

x_min = dates.iloc[start_idx]
x_max = dates.iloc[end_idx]

visible_prices = prices.iloc[start_idx : end_idx + 1]

price_padding = (visible_prices.max() - visible_prices.min()) * 0.12
if price_padding == 0:
    price_padding = visible_prices.iloc[0] * 0.05

price_y_min = min(
    visible_prices.min(),
    effective_convexity_basis.iloc[start_idx : end_idx + 1].min(),
    basis_2022,
    basis_2023,
) - price_padding

price_y_max = max(
    visible_prices.max(),
    basis_2022,
    basis_2023,
) + price_padding

portfolio_max = np.nanmax([
    value_2022.max(),
    value_2023.max(),
    convexity_value.max(),
])

portfolio_min = np.nanmin([
    value_2022.min(),
    value_2023.min(),
    convexity_value.min(),
])

portfolio_padding = (portfolio_max - portfolio_min) * 0.12

portfolio_y_min = portfolio_min - portfolio_padding
portfolio_y_max = portfolio_max + portfolio_padding

right_bar_max = max(
    basis_2022,
    basis_2023,
    np.nanmax(effective_convexity_basis),
    np.nanmax(value_2022),
    np.nanmax(value_2023),
    np.nanmax(convexity_value),
)

# ==========================================================
# DATE ANNOTATION
# ==========================================================

def date_annotation(i):
    return dict(
        x=0.99,
        y=-0.055,
        xref="paper",
        yref="paper",
        showarrow=False,
        xanchor="right",
        yanchor="top",
        font=dict(color=TEXT, size=14),
        text=f"Date: {dates.iloc[i].strftime('%b %d, %Y')}",
    )

# ==========================================================
# FIGURE
# ==========================================================

fig = make_subplots(
    rows=2,
    cols=2,
    column_widths=[0.70, 0.30],
    row_heights=[0.55, 0.45],
    horizontal_spacing=0.07,
    vertical_spacing=0.14,
    specs=[
        [{"type": "xy"}, {"type": "table"}],
        [{"type": "xy"}, {"type": "xy"}],
    ],
    subplot_titles=[
        "SPY price and effective cost basis",
        "",
        "Portfolio value path",
        "Cost basis and terminal value",
    ],
)

# ==========================================================
# TOP LEFT: SPY PRICE + BASIS LINES
# ==========================================================

fig.add_trace(
    go.Scatter(
        x=[dates.iloc[start_idx]],
        y=[prices.iloc[start_idx]],
        mode="lines",
        line=dict(color=PRICE_COLOR, width=PRICE_WIDTH),
        name="SPY close",
        hovertemplate="Date: %{x|%b %d, %Y}<br>SPY: $%{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=[dates.iloc[start_idx]],
        y=[prices.iloc[start_idx]],
        mode="markers",
        marker=dict(size=MARKER_SIZE, color=PRICE_COLOR),
        showlegend=False,
        hovertemplate="Date: %{x|%b %d, %Y}<br>SPY: $%{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=[entry_date_2022, x_max],
        y=[basis_2022, basis_2022],
        mode="lines",
        line=dict(color=BH_COLOR, width=BASIS_WIDTH, dash="dash"),
        name="Jan 2022 basis",
        hovertemplate=f"Jan 2022 basis: ${basis_2022:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=[entry_date_2023, entry_date_2023],
        y=[basis_2023, basis_2023],
        mode="lines",
        line=dict(color=TIMING_COLOR, width=BASIS_WIDTH, dash="dash"),
        name="Jan 2023 timing basis",
        hovertemplate=f"Jan 2023 basis: ${basis_2023:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=[entry_date_2022],
        y=[effective_convexity_basis.iloc[start_idx]],
        mode="lines",
        line=dict(color=CONVEXITY_COLOR, width=2.7),
        name="Effective basis after put monetization",
        hovertemplate="Date: %{x|%b %d, %Y}<br>Effective basis: $%{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

# ==========================================================
# TOP RIGHT: TABLE
# ==========================================================

fig.add_trace(
    make_stats_table(stats_for(start_idx)),
    row=1,
    col=2,
)

# ==========================================================
# BOTTOM LEFT: PORTFOLIO VALUE PATHS
# ==========================================================

fig.add_trace(
    go.Scatter(
        x=[dates.iloc[start_idx]],
        y=[value_2022.iloc[start_idx]],
        mode="lines",
        line=dict(color=BH_COLOR, width=PORTFOLIO_WIDTH),
        name=LABEL_2022,
        hovertemplate="Date: %{x|%b %d, %Y}<br>Value: $%{y:,.0f}<extra></extra>",
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=[entry_date_2023],
        y=[np.nan],
        mode="lines",
        line=dict(color=TIMING_COLOR, width=PORTFOLIO_WIDTH),
        name=LABEL_2023,
        hovertemplate="Date: %{x|%b %d, %Y}<br>Value: $%{y:,.0f}<extra></extra>",
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=[dates.iloc[start_idx]],
        y=[convexity_value.iloc[start_idx]],
        mode="lines",
        line=dict(color=CONVEXITY_COLOR, width=PORTFOLIO_WIDTH),
        name=LABEL_CONVEXITY,
        hovertemplate="Date: %{x|%b %d, %Y}<br>Value: $%{y:,.0f}<extra></extra>",
    ),
    row=2,
    col=1,
)

# ==========================================================
# BOTTOM RIGHT: BARS
# ==========================================================

initial_stats = stats_for(start_idx)

fig.add_trace(
    go.Bar(
        x=[LABEL_2022, LABEL_2023, LABEL_CONVEXITY],
        y=[
            basis_2022,
            basis_2023,
            initial_stats["effective_basis"],
        ],
        name="Cost basis",
        marker=dict(color=[BH_COLOR, TIMING_COLOR, CONVEXITY_COLOR]),
        opacity=0.55,
        hovertemplate="%{x}<br>Cost basis: $%{y:.2f}<extra></extra>",
    ),
    row=2,
    col=2,
)

fig.add_trace(
    go.Bar(
        x=[LABEL_2022, LABEL_2023, LABEL_CONVEXITY],
        y=[
            initial_stats["bh_terminal"],
            0,
            initial_stats["convexity_terminal"],
        ],
        name="Terminal value",
        marker=dict(color=[BH_COLOR, TIMING_COLOR, CONVEXITY_COLOR]),
        opacity=1.0,
        hovertemplate="%{x}<br>Terminal value: $%{y:,.0f}<extra></extra>",
    ),
    row=2,
    col=2,
)

# Trace indexes:
# 0 SPY close
# 1 SPY endpoint
# 2 Jan 2022 basis
# 3 Jan 2023 timing basis
# 4 Convexity effective basis
# 5 Stats table
# 6 Jan 2022 B&H value
# 7 Jan 2023 timing value
# 8 Convexity value
# 9 Cost basis bars
# 10 Terminal value bars

# ==========================================================
# FRAMES
# ==========================================================

frames = []

for i in frame_indices:
    frame_name = f"frame_{i}"
    s = stats_for(i)

    price_x = dates.iloc[start_idx : i + 1]
    price_y = prices.iloc[start_idx : i + 1]

    convexity_basis_x = dates.iloc[start_idx : i + 1]
    convexity_basis_y = effective_convexity_basis.iloc[start_idx : i + 1]

    bh_value_x = dates.iloc[idx_2022 : i + 1]
    bh_value_y = value_2022.iloc[idx_2022 : i + 1]

    convexity_value_x = dates.iloc[idx_2022 : i + 1]
    convexity_value_y = convexity_value.iloc[idx_2022 : i + 1]

    if i >= idx_2023:
        timing_basis_x = [entry_date_2023, dates.iloc[i]]
        timing_basis_y = [basis_2023, basis_2023]

        timing_value_x = dates.iloc[idx_2023 : i + 1]
        timing_value_y = value_2023.iloc[idx_2023 : i + 1]

        timing_terminal_bar = s["timing_terminal"]
    else:
        timing_basis_x = [entry_date_2023, entry_date_2023]
        timing_basis_y = [basis_2023, basis_2023]

        timing_value_x = [entry_date_2023]
        timing_value_y = [np.nan]

        timing_terminal_bar = 0

    frames.append(
        go.Frame(
            name=frame_name,
            traces=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            data=[
                go.Scatter(
                    x=price_x,
                    y=price_y,
                ),
                go.Scatter(
                    x=[dates.iloc[i]],
                    y=[prices.iloc[i]],
                ),
                go.Scatter(
                    x=[entry_date_2022, dates.iloc[i]],
                    y=[basis_2022, basis_2022],
                ),
                go.Scatter(
                    x=timing_basis_x,
                    y=timing_basis_y,
                ),
                go.Scatter(
                    x=convexity_basis_x,
                    y=convexity_basis_y,
                ),
                make_stats_table(s),
                go.Scatter(
                    x=bh_value_x,
                    y=bh_value_y,
                ),
                go.Scatter(
                    x=timing_value_x,
                    y=timing_value_y,
                ),
                go.Scatter(
                    x=convexity_value_x,
                    y=convexity_value_y,
                ),
                go.Bar(
                    x=[LABEL_2022, LABEL_2023, LABEL_CONVEXITY],
                    y=[
                        basis_2022,
                        basis_2023,
                        s["effective_basis"],
                    ],
                ),
                go.Bar(
                    x=[LABEL_2022, LABEL_2023, LABEL_CONVEXITY],
                    y=[
                        s["bh_terminal"],
                        timing_terminal_bar,
                        s["convexity_terminal"],
                    ],
                ),
            ],
            layout=go.Layout(
                annotations=[
                    date_annotation(i),
                ]
            ),
        )
    )

fig.frames = frames

# ==========================================================
# SLIDER STEPS — ONE PER YEAR ONLY
# ==========================================================

slider_steps = []

for i in year_step_indices:
    frame_name = f"frame_{i}"

    slider_steps.append(
        dict(
            args=[
                [frame_name],
                dict(
                    frame=dict(duration=0, redraw=True),
                    transition=dict(duration=0),
                    mode="immediate",
                ),
            ],
            label=dates.iloc[i].strftime("%Y"),
            method="animate",
        )
    )

# ==========================================================
# LAYOUT
# ==========================================================

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    width=1450,
    height=850,
    margin=dict(l=70, r=45, t=95, b=155),
    barmode="group",
    showlegend=False,
    title=dict(
        text="SPY: Monetized Long Put Convexity vs Waiting for a Better Entry",
        x=0.04,
        y=0.985,
        xanchor="left",
        font=dict(size=27, color=TEXT),
    ),
    annotations=[
        date_annotation(start_idx),
    ],
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            showactive=False,
            x=0.00,
            y=-0.18,
            xanchor="left",
            yanchor="top",
            pad=dict(r=10, t=30),
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(duration=ANIMATION_SPEED, redraw=True),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                ),
                dict(
                    label="⏸ Pause",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            frame=dict(duration=0, redraw=True),
                            transition=dict(duration=0),
                            mode="immediate",
                        ),
                    ],
                ),
            ],
        )
    ],
    sliders=[
        dict(
            active=0,
            x=0.19,
            y=-0.165,
            len=0.75,
            xanchor="left",
            yanchor="top",
            pad=dict(t=45, b=10),
            currentvalue=dict(visible=False),
            transition=dict(duration=0),
            steps=slider_steps,
        )
    ],
)

# ==========================================================
# AXES
# ==========================================================

fig.update_xaxes(
    row=1,
    col=1,
    range=[x_min, x_max],
    title_text="Date",
    title_font=dict(size=14, color=TEXT),
    tickfont=dict(size=11, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_yaxes(
    row=1,
    col=1,
    range=[price_y_min, price_y_max],
    title_text="SPY Price / Effective Basis ($)",
    title_font=dict(size=14, color=TEXT),
    tickfont=dict(size=11, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_xaxes(
    row=2,
    col=1,
    range=[x_min, x_max],
    title_text="Date",
    title_font=dict(size=14, color=TEXT),
    tickfont=dict(size=11, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_yaxes(
    row=2,
    col=1,
    range=[portfolio_y_min, portfolio_y_max],
    title_text="Portfolio Value ($)",
    title_font=dict(size=14, color=TEXT),
    tickfont=dict(size=11, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_xaxes(
    row=2,
    col=2,
    tickfont=dict(size=10, color=TEXT),
    showgrid=False,
    linecolor=TEXT,
)

fig.update_yaxes(
    row=2,
    col=2,
    range=[0, right_bar_max * 1.15],
    title_text="$",
    title_font=dict(size=14, color=TEXT),
    tickfont=dict(size=11, color=TEXT),
    showgrid=True,
    gridcolor=GRID,
    linecolor=TEXT,
    zeroline=False,
)

fig.update_xaxes(row=1, col=2, visible=False)
fig.update_yaxes(row=1, col=2, visible=False)

fig.show()

# Optional
# fig.write_html("spy_put_reinvestment_vs_timing_fixed.html", auto_open=True)

---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary**

  - This lesson explored how overlaying a long volatility component onto a portfolio can actively combat volatility drag and make portfolios more resilient during market shocks and drawdowns.
  - We demonstrated that, compared to traditional "timing" approaches, a capital-efficient long vol overlay can not only smooth the ride through turbulent markets but actually enhance overall risk-adjusted returns—even when the market itself goes sideways or whipsaws.
  - Through code and an animated Plotly visualization, we quantified the impact: showing how the growth of $1, max drawdown, and performance ratios (Sharpe, Sortino, etc.) all improve with an intelligently constructed vol overlay as compared to simple market exposure.
  - Ultimately, you saw how quants use these techniques to go beyond trying (and failing) to "time the market," instead engineering portfolios that systematically benefit from volatility, turning a supposed drag into an edge.

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$